In [0]:
%python
import subprocess
import sys

try:
    spark.sql("CREATE CATALOG IF NOT EXISTS supply_chain_dev")
    spark.sql("USE CATALOG supply_chain_dev")
    print("Unity Catalog available — using supply_chain_dev catalog")
except Exception:
    print("Unity Catalog not available — using hive_metastore")

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS shipments
    COMMENT 'Supply Chain Event Sourcing Project'
""")

spark.sql("USE SCHEMA shipments")
print("Schema ready: shipments")

Unity Catalog available — using supply_chain_dev catalog
Schema ready: shipments


In [0]:
%python
BASE_PATH = "dbfs:/supply_chain_project"

dbutils.fs.mkdirs(f"{BASE_PATH}/raw/shipment_events")
dbutils.fs.mkdirs(f"{BASE_PATH}/raw/carrier_updates")
dbutils.fs.mkdirs(f"{BASE_PATH}/checkpoints/bronze")
dbutils.fs.mkdirs(f"{BASE_PATH}/checkpoints/silver")
dbutils.fs.mkdirs(f"{BASE_PATH}/data/bronze")
dbutils.fs.mkdirs(f"{BASE_PATH}/data/silver")
dbutils.fs.mkdirs(f"{BASE_PATH}/data/gold")

display(dbutils.fs.ls(f"{BASE_PATH}/"))
print("All directories created successfully")

---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-4922481868675123>, line 3
      1 BASE_PATH = "dbfs:/supply_chain_project"
----> 3 dbutils.fs.mkdirs(f"{BASE_PATH}/raw/shipment_events")
      4 dbutils.fs.mkdirs(f"{BASE_PATH}/raw/carrier_updates")
      5 dbutils.fs.mkdirs(f"{BASE_PATH}/checkpoints/bronze")

File /databricks/python_shell/lib/dbruntime/remotefshandler/RemoteFsHandler.py:56, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
     52     pass
     54 error_exception = ExecutionError(str(e))
---> 56 raise patch_exception_with_error_details(
     57     error_exception,
     58     DriverErrorCode.REMOTE_FS_HANDLER_EXECUTION_ERROR  # type: ignore[attr-defined]
     59 ) from None

ExecutionError: [DBFS_DISABLED] Public DBFS root is disabled. Access is denied on path: /supply_chain_project/raw/shipment_events SQLSTATE: 56038

JVM

In [0]:
%python
# Unity Catalog Volumes = modern replacement for DBFS
# Format: /Volumes/catalog/schema/volume_name/

spark.sql("""
    CREATE VOLUME IF NOT EXISTS supply_chain_dev.shipments.raw_files
    COMMENT 'Landing zone for raw shipment JSON files'
""")

print("Volume created: supply_chain_dev.shipments.raw_files")

BASE_PATH = "/Volumes/supply_chain_dev/shipments/raw_files"

import os
os.makedirs(f"{BASE_PATH}/raw/shipment_events", exist_ok=True)
os.makedirs(f"{BASE_PATH}/raw/carrier_updates", exist_ok=True)
os.makedirs(f"{BASE_PATH}/checkpoints/bronze", exist_ok=True)
os.makedirs(f"{BASE_PATH}/checkpoints/silver", exist_ok=True)

print("All directories created successfully")
print(f"Base path: {BASE_PATH}")

Volume created: supply_chain_dev.shipments.raw_files
All directories created successfully
Base path: /Volumes/supply_chain_dev/shipments/raw_files


In [0]:
%python
%pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.7 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%python
dbutils.library.restartPython()

In [0]:
%python
import json
import random
from datetime import datetime, timedelta
from faker import Faker
import uuid

fake = Faker()
random.seed(42)

CARRIERS = ["CMA CGM", "Hapag-Lloyd", "MSC", "Evergreen", "COSCO"]
PORT_CODES = ["USLAX", "CNSHA", "SGSIN", "NLRTM", "DEHAM", "AEJEA", "HKHKG", "KRPUS"]
STATUSES = [
    "BOOKING_CONFIRMED", "CARGO_RECEIVED", "VESSEL_DEPARTED",
    "IN_TRANSIT", "PORT_ARRIVED", "CUSTOMS_HOLD",
    "CUSTOMS_RELEASED", "OUT_FOR_DELIVERY", "DELIVERED", "EXCEPTION"
]
COMMODITY_TYPES = ["Electronics", "Apparel", "Auto Parts", "Chemicals", "Machinery"]

def generate_booking_id():
    prefix = random.choice(["CMA", "HAP", "MSC", "EVG", "COS"])
    return f"{prefix}{random.randint(100000000, 999999999)}"

def generate_container_id():
    letters = ''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ', k=4))
    return f"{letters}{random.randint(1000000, 9999999)}"

def generate_shipment_event(booking_id, container_id, carrier, origin, destination,
                             status, event_dt, prev_status=None, is_corrupt=False):
    event = {
        "event_id": str(uuid.uuid4()),
        "booking_id": booking_id,
        "container_id": container_id,
        "carrier": carrier,
        "origin_port": origin,
        "destination_port": destination,
        "status": status,
        "previous_status": prev_status,
        "event_timestamp": event_dt.isoformat(),
        "vessel_name": fake.bothify(text="MV ????-???").upper() if status in [
            "VESSEL_DEPARTED", "IN_TRANSIT", "PORT_ARRIVED"
        ] else None,
        "commodity_type": random.choice(COMMODITY_TYPES),
        "weight_kg": round(random.uniform(500, 25000), 2),
        "estimated_arrival": (event_dt + timedelta(days=random.randint(14, 45))).isoformat(),
        "exception_reason": fake.sentence() if status == "EXCEPTION" else None,
        "source_system": random.choice(["CargoWise", "SAP_TM", "Inttra", "GT_Nexus"]),
        "schema_version": "1.0"
    }
    if is_corrupt:
        corrupt_type = random.choice(["null_booking", "null_status", "bad_weight", "missing_ts"])
        if corrupt_type == "null_booking":
            event["booking_id"] = None
        elif corrupt_type == "null_status":
            event["status"] = None
        elif corrupt_type == "bad_weight":
            event["weight_kg"] = -99.0
        elif corrupt_type == "missing_ts":
            event["event_timestamp"] = None
    return event

print("All functions defined successfully!")

All functions defined successfully!


In [0]:
%python
BASE_PATH = "/Volumes/supply_chain_dev/shipments/raw_files"

shipments = []
for _ in range(80):
    origin, destination = random.sample(PORT_CODES, 2)
    shipments.append({
        "booking_id": generate_booking_id(),
        "container_id": generate_container_id(),
        "carrier": random.choice(CARRIERS),
        "origin": origin,
        "destination": destination
    })

events_batch1 = []
base_date = datetime(2024, 1, 1)

for shipment in shipments:
    current_dt = base_date + timedelta(days=random.randint(0, 30))
    prev_status = None
    num_events = random.randint(4, 8)
    statuses_for_shipment = STATUSES[:num_events]

    for status in statuses_for_shipment:
        is_corrupt = random.random() < 0.08
        event = generate_shipment_event(
            booking_id=shipment["booking_id"],
            container_id=shipment["container_id"],
            carrier=shipment["carrier"],
            origin=shipment["origin"],
            destination=shipment["destination"],
            status=status,
            event_dt=current_dt,
            prev_status=prev_status,
            is_corrupt=is_corrupt
        )
        events_batch1.append(event)
        prev_status = status
        current_dt += timedelta(hours=random.randint(6, 72))

# Write to files
batch_size = 50
file_count = 0
for i in range(0, len(events_batch1), batch_size):
    chunk = events_batch1[i:i+batch_size]
    filename = f"{BASE_PATH}/raw/shipment_events/batch1_file{file_count:03d}.json"
    with open(filename, 'w') as f:
        for event in chunk:
            f.write(json.dumps(event) + '\n')
    file_count += 1

print(f"Total events generated: {len(events_batch1)}")
print(f"Total files written: {file_count}")
print(f"Location: {BASE_PATH}/raw/shipment_events/")

Total events generated: 466
Total files written: 10
Location: /Volumes/supply_chain_dev/shipments/raw_files/raw/shipment_events/


In [0]:
# Read one file back and preview it
sample_events = []
with open(f"{BASE_PATH}/raw/shipment_events/batch1_file000.json") as f:
    for line in f:
        sample_events.append(json.loads(line))

# Convert to Spark DataFrame for clean display
sample_df = spark.createDataFrame(sample_events)

print(f"Total records in file: {len(sample_events)}")
print("\nSchema:")
sample_df.printSchema()

print("\nSample records:")
display(sample_df.select(
    "booking_id", "carrier", "origin_port", 
    "destination_port", "status", "weight_kg"
).limit(10))

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4922481868675129>, line 3
      1 # Read one file back and preview it
      2 sample_events = []
----> 3 with open(f"{BASE_PATH}/raw/shipment_events/batch1_file000.json") as f:
      4     for line in f:
      5         sample_events.append(json.loads(line))

NameError: name 'BASE_PATH' is not defined

In [0]:
import json

BASE_PATH = "/Volumes/supply_chain_dev/shipments/raw_files"

# Read one file back and preview it
sample_events = []
with open(f"{BASE_PATH}/raw/shipment_events/batch1_file000.json") as f:
    for line in f:
        sample_events.append(json.loads(line))

# Convert to Spark DataFrame for clean display
sample_df = spark.createDataFrame(sample_events)

print(f"Total records in file: {len(sample_events)}")
print("\nSchema:")
sample_df.printSchema()

print("\nSample records:")
display(sample_df.select(
    "booking_id", "carrier", "origin_port", 
    "destination_port", "status", "weight_kg"
).limit(10))

---------------------------------------------------------------------------
PySparkValueError                         Traceback (most recent call last)
File <command-4922481868675130>, line 12
      9         sample_events.append(json.loads(line))
     11 # Convert to Spark DataFrame for clean display
---> 12 sample_df = spark.createDataFrame(sample_events)
     14 print(f"Total records in file: {len(sample_events)}")
     15 print("\nSchema:")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:811, in SparkSession.createDataFrame(self, data, schema, samplingRatio, verifySchema)
    806         _num_cols = len(_cols)
    808     if _has_nulltype(_schema):
    809         # For cases like createDataFrame([("Alice", None, 80.1)], schema)
    810         # we can not infer the schema from the data itself.
--> 811         raise PySparkValueError(
    812             errorClass="CANNOT_DETERMINE_TYPE", messageParameters={}
    813         )
    815 from pys

In [0]:
import json
from pyspark.sql.types import *

BASE_PATH = "/Volumes/supply_chain_dev/shipments/raw_files"

# Define schema explicitly instead of letting Spark infer it
schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("booking_id", StringType(), True),
    StructField("container_id", StringType(), True),
    StructField("carrier", StringType(), True),
    StructField("origin_port", StringType(), True),
    StructField("destination_port", StringType(), True),
    StructField("status", StringType(), True),
    StructField("previous_status", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("vessel_name", StringType(), True),
    StructField("commodity_type", StringType(), True),
    StructField("weight_kg", DoubleType(), True),
    StructField("estimated_arrival", StringType(), True),
    StructField("exception_reason", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("schema_version", StringType(), True)
])

sample_events = []
with open(f"{BASE_PATH}/raw/shipment_events/batch1_file000.json") as f:
    for line in f:
        sample_events.append(json.loads(line))

sample_df = spark.createDataFrame(sample_events, schema=schema)

print(f"Total records in file: {len(sample_events)}")
display(sample_df.select(
    "booking_id", "carrier", "origin_port",
    "destination_port", "status", "weight_kg"
).limit(10))

Total records in file: 50


booking_id,carrier,origin_port,destination_port,status,weight_kg
HAP919795579,CMA CGM,DEHAM,CNSHA,BOOKING_CONFIRMED,2737.51
HAP919795579,CMA CGM,DEHAM,CNSHA,CARGO_RECEIVED,12027.27
HAP919795579,CMA CGM,DEHAM,CNSHA,VESSEL_DEPARTED,16052.23
HAP919795579,CMA CGM,DEHAM,CNSHA,IN_TRANSIT,8365.93
HAP919795579,CMA CGM,DEHAM,CNSHA,PORT_ARRIVED,7561.34
HAP919795579,CMA CGM,DEHAM,CNSHA,CUSTOMS_HOLD,8420.21
CMA506448196,COSCO,KRPUS,DEHAM,BOOKING_CONFIRMED,15754.53
CMA506448196,COSCO,KRPUS,DEHAM,CARGO_RECEIVED,3709.82
CMA506448196,COSCO,KRPUS,DEHAM,VESSEL_DEPARTED,20083.19
CMA506448196,COSCO,KRPUS,DEHAM,IN_TRANSIT,6005.17


In [0]:
from pyspark.sql.functions import col

corrupted = sample_df.filter(
    col("booking_id").isNull() |
    col("status").isNull() |
    (col("weight_kg") < 0) |
    col("event_timestamp").isNull()
)

print(f"Clean records: {sample_df.count() - corrupted.count()}")
print(f"Corrupted records: {corrupted.count()}")
display(corrupted.select("booking_id", "status", "weight_kg", "event_timestamp"))

Clean records: 49
Corrupted records: 1


booking_id,status,weight_kg,event_timestamp
MSC274648506,CUSTOMS_HOLD,-99.0,2024-01-25T02:00:00
